In [ ]:
%matplotlib widget
import numpy 
import pandas
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets
import os
import cryspy
from ipyfilechooser import FileChooser
from magic_tof import model_MAGiC as model_magic
import scipp 

In [ ]:
def calc_gamma_nu_wavelength_for_hkl(h, k, l, UB, R):
    hkl = numpy.vstack([h, k, l]).T
    Q = (R @ (UB @ hkl.T)).T
    Qnorm = numpy.linalg.norm(Q, axis=1)
    cos_alpha = -Q[:,2]/Qnorm
    wavelength = 2 * cos_alpha / Qnorm # 4*numpy.pi
    ki = numpy.zeros(Q.shape,dtype=float)
    ki[:,2] = 1/wavelength # 2*numpy.pi
    kf = ki + Q
    kf_x, kf_y, kf_z = kf[:,0], kf[:,1], kf[:,2]
    
    r = numpy.linalg.norm(kf, axis=1)

    gamma = numpy.rad2deg(numpy.arctan2(kf_x, kf_z))      # horizontal angle
    nu    = numpy.rad2deg(numpy.arcsin(kf_y / r))  
    return gamma, nu, wavelength


In [ ]:
def generate_peak_data(UB, R, hmax, kmax, lmax, lambda_min, lambda_max):
    """
    Generate synthetic diffraction peak data based on:
    - UB matrix (3x3)
    - crystal rotation matrix R (3x3)
    - HKL range: -hmax..hmax etc.
    - wavelength range (lambda_min, lambda_max)

    Returns array with columns:
    [h, k, l, gamma_deg, nu_deg, wavelength]
    """

    # --- 1. Generate HKL grid ---
    h = numpy.arange(-int(hmax), int(hmax+1))
    k = numpy.arange(-int(kmax), int(kmax+1))
    l = numpy.arange(-int(lmax), int(lmax+1))
    H, K, L = numpy.meshgrid(h, k, l, indexing='ij')
    hkl = numpy.vstack([H.ravel(), K.ravel(), L.ravel()]).T

    # Remove (0,0,0)
    hkl = hkl[numpy.any(hkl != 0, axis=1)]

    # --- 2. Compute Q vectors in lab frame ---
    # Apply UB and then rotation R
    Q = (R @ (UB @ hkl.T)).T

    # Magnitude of Q
    Qnorm = numpy.linalg.norm(Q, axis=1)


    # --- 3. Compute wavelength from Bragg condition ---
    cos_alpha = -Q[:,2]/Qnorm

    wavelength = 2 * cos_alpha / Qnorm # 4*numpy.pi

    # wavelength = 4*numpy.pi / Qnorm

    # --- 4. Apply wavelength limits ---
    mask = (wavelength >= lambda_min) & (wavelength <= lambda_max)
    hkl = hkl[mask]
    Q = Q[mask]
    wavelength = wavelength[mask]

    # --- 5. Convert Q direction to detector angles (γ, ν) ---
    ki = numpy.zeros(Q.shape,dtype=float)
    ki[:,2] = 1/wavelength # 2*numpy.pi
    kf = ki + Q
    kf_x, kf_y, kf_z = kf[:,0], kf[:,1], kf[:,2]
    
    r = numpy.linalg.norm(kf, axis=1)

    gamma = numpy.rad2deg(numpy.arctan2(kf_x, kf_z))      # horizontal angle
    nu    = numpy.rad2deg(numpy.arcsin(kf_y / r))       # vertical angle

    # --- 6. Build final array ---
    result = numpy.column_stack([hkl[:,0], hkl[:,1], hkl[:,2], gamma, nu, wavelength])
    return result




In [ ]:
def calc_orientation_matrix(euler_alpha, euler_beta, euler_gamma, ):
    ca, cb, cg = numpy.cos(euler_alpha), numpy.cos(euler_beta), numpy.cos(euler_gamma)
    sa, sb, sg = numpy.sin(euler_alpha), numpy.sin(euler_beta), numpy.sin(euler_gamma)
    m_m = numpy.array([
        [ca*cb, ca*sb*sg-sa*cg, ca*sb*cg+sa*sg],
        [sa*cb, sa*sb*sg+ca*cg, sa*sb*cg-ca*sg],
        [-sb, cb*sg, cb*cg],
    ], dtype=float)
    return m_m

In [ ]:

def calc_sample_rotation(sample_omega, sample_chi, sample_phi):
    zero_o = numpy.sin(numpy.zeros_like(sample_omega))
    one_o = numpy.cos(numpy.zeros_like(sample_omega))
    m_omega = numpy.array([
        [numpy.cos(sample_omega), zero_o, numpy.sin(sample_omega)],
        [zero_o, one_o, zero_o],
        [-numpy.sin(sample_omega), zero_o, numpy.cos(sample_omega)],
    ],dtype=float)
    zero_c = numpy.sin(numpy.zeros_like(sample_chi))
    one_c = numpy.cos(numpy.zeros_like(sample_chi))
    m_chi = numpy.array([
        [numpy.cos(sample_chi), -numpy.sin(sample_chi), zero_c],
        [numpy.sin(sample_chi), numpy.cos(sample_chi), zero_c],
        [zero_c, zero_c, one_c],
    ],dtype=float)

    zero_p = numpy.sin(numpy.zeros_like(sample_phi))
    one_p = numpy.cos(numpy.zeros_like(sample_phi))
    m_phi = numpy.array([
        [numpy.cos(sample_phi), zero_p, numpy.sin(sample_phi)],
        [zero_p, one_p, zero_p],
        [-numpy.sin(sample_phi), zero_p, numpy.cos(sample_phi)],
    ], dtype=float)
    sample_rotation = m_omega @ m_chi @ m_phi
    return sample_rotation


In [ ]:

def _calc_cell_phi(cell_alpha, cell_beta, cell_gamma):
    ca, cb, cg = numpy.cos(cell_alpha), numpy.cos(cell_beta), numpy.cos(cell_gamma)
    cell_phi = numpy.sqrt(1. - ca*ca - cb*cb - cg*cg + 2 * ca * cb * cg)
    return cell_phi

def calc_cell_volume(cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma):
    cell_phi = _calc_cell_phi(cell_alpha, cell_beta, cell_gamma)
    cell_volume = cell_a * cell_b * cell_c * cell_phi
    return cell_volume

def calc_b_matrix(cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma):
    cell_phi = _calc_cell_phi(cell_alpha, cell_beta, cell_gamma)
    a, b, c = cell_a, cell_b, cell_c
    b_11 = numpy.sin(cell_alpha)/(a*cell_phi)
    b_12 = (numpy.cos(cell_alpha)*numpy.cos(cell_beta)-numpy.cos(cell_gamma))/(b*cell_phi*numpy.sin(cell_alpha))
    b_13 = (numpy.cos(cell_alpha)*numpy.cos(cell_gamma)-numpy.cos(cell_beta))/(c*cell_phi*numpy.sin(cell_alpha))
    b_22 = 1/(b*numpy.sin(cell_alpha))
    b_23 = -numpy.cos(cell_alpha)/(c*numpy.sin(cell_alpha))
    b_33 = 1/c
    zero = 0.
    b_matrix = numpy.array([
            [b_11, b_12, b_13],
            [zero, b_22, b_23],
            [zero, zero, b_33],
        ],dtype=float)
    return b_matrix

In [ ]:
def hkl_to_rgb(h,k,l):
    hkl = numpy.sqrt(numpy.square(h) + numpy.square(k) + numpy.square(l))
    hh_r = numpy.abs(h) / hkl
    hh_g = numpy.abs(k) / hkl
    hh_b = numpy.abs(l) / hkl
    return numpy.stack([hh_r, hh_g, hh_b], axis=1)

In [ ]:
def simulate_detector(intensities, gammas, nus, 
                      gamma_grid, nu_grid, sigma):
    """
    Simulate a diffraction pattern on a 2D detector using Gaussian peaks.

    Parameters
    ----------
    intensities : array-like
        Integrated intensities of reflections (I_i).
    gammas : array-like
        Gamma positions of reflections.
    nus : array-like
        Nu positions of reflections.
    gamma_grid : 2D ndarray
        Meshgrid of gamma coordinates on detector.
    nu_grid : 2D ndarray
        Meshgrid of nu coordinates on detector.
    sigma : float
        Gaussian peak width (constant).

    Returns
    -------
    pattern : 2D ndarray
        Simulated detector intensity.
    """

    pattern = numpy.zeros_like(gamma_grid, dtype=float)

    for I, g0, n0 in zip(intensities, gammas, nus):
        # Gaussian peak
        pattern += I * numpy.exp(
            -((gamma_grid - g0)**2 + (nu_grid - n0)**2) / (2 * sigma**2)
        )

    return pattern


from matplotlib.colors import LogNorm

def plot_pattern(pattern, gamma_grid, nu_grid):
    # Determine axis limits from the meshgrid
    pattern = numpy.clip(pattern, 1e-12, None)
    gamma_min, gamma_max = gamma_grid.min(), gamma_grid.max()
    nu_min, nu_max = nu_grid.min(), nu_grid.max()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(
        pattern,
        extent=[gamma_min, gamma_max, nu_min, nu_max],
        origin="lower",
        # cmap="inferno",
        aspect="auto",
        norm=LogNorm(vmin=pattern.max()*1e-4, vmax=pattern.max()) 
    )
    fig.colorbar(im, ax=ax, label="Intensity")
    ax.set_xlabel("gamma")
    ax.set_ylabel("nu")
    ax.set_title("Simulated Diffraction Pattern")
    return fig



In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

def plot_scatters_with_labels(x, y, labels, color='red', size=4, n_visible=10, fig=None, color_text='red', marker='o'):
    if fig is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    else:
        ax = fig.axes[0]
    sc = ax.scatter(x, y, s=size, color=color,marker=marker)

    active_texts = []

    x = np.asarray(x)
    y = np.asarray(y)
    labels = np.asarray(labels)

    def update_labels(event=None):
        nonlocal active_texts

        # Remove old labels
        for t in active_texts:
            t.remove()
        active_texts = []

        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()

        mask = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        idx = np.where(mask)[0]

        # Only label if few points are visible
        if len(idx) <= n_visible:

            # --- GROUP POINTS BY (x, y) ---
            groups = {}
            for i in idx:
                key = (numpy.round(x[i],3), numpy.round(y[i],3))
                if key not in groups:
                    groups[key] = []
                groups[key].append(labels[i])

            # --- CREATE ONE LABEL PER GROUP ---
            for (xi, yi), labs in groups.items():
                # Split into chunks of 5 labels
                chunks = [labs[i:i+5] for i in range(0, len(labs), 5)]
                # Join each chunk with commas, then join chunks with newlines
                merged_label = "\n".join(", ".join(chunk) for chunk in chunks)                
                # merged_label = ", ".join(labs)
                t = ax.text(xi, yi, merged_label,
                            fontsize=8, color=color_text)
                active_texts.append(t)

        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("draw_event", update_labels)
    fig.canvas.mpl_connect("button_release_event", update_labels)
    fig.canvas.mpl_connect("scroll_event", update_labels)

    update_labels()
    return fig

# # Example
# np.random.seed(0)
# N = 20000
# x = np.random.randn(N)
# y = np.random.randn(N)
# labels = np.array([f"P{i}" for i in range(N)])

# fig = plot_scatters_with_labels(x, y, labels, n_visible=10)
# widgets.VBox([fig.canvas])

In [ ]:
%matplotlib widget

# ---------------------------
# Widgets
# ---------------------------
dir_path_data = "/Users/iuriikibalin/Desktop"

if not os.path.isdir(dir_path_data):
    dir_path_data = "/"

fc = FileChooser(
    dir_path_data,
    title="Select CIF File (Optional)",
    select_default=False,
)

cell_a_w = ipywidgets.FloatText(description="a", value=14.0)
cell_b_w = ipywidgets.FloatText(description="b", value=14.0)
cell_c_w = ipywidgets.FloatText(description="c", value=14.0)

cell_alpha_w = ipywidgets.FloatText(description="α (deg)", value=90.0)
cell_beta_w  = ipywidgets.FloatText(description="β (deg)", value=90.0)
cell_gamma_w = ipywidgets.FloatText(description="γ (deg)", value=90.0)


gamma_min_w = ipywidgets.FloatText(description="γ_min", value=10)
gamma_max_w = ipywidgets.FloatText(description="γ_max", value=70)

nu_min_w = ipywidgets.FloatText(description="ν_min", value=-35)
nu_max_w = ipywidgets.FloatText(description="ν_max", value=15)

ea_alpha_w = ipywidgets.FloatText(description="α (deg)", value=0.0)
ea_beta_w  = ipywidgets.FloatText(description="β (deg)", value=0.0)
ea_gamma_w = ipywidgets.FloatText(description="γ (deg)", value=0.0)

omega_w = ipywidgets.FloatText(description="ω (deg)", value=0.0)
chi_w   = ipywidgets.FloatText(description="χ (deg)", value=0.0)
phi_w   = ipywidgets.FloatText(description="φ (deg)", value=0.0)



psc_nu_dropdown = ipywidgets.Dropdown(description="PSC_nu", options=[14,28,70,112, 154], value=154, disabled=False)
psc_opening_angle_floattext = ipywidgets.BoundedFloatText(description="Opening angle of PSC", min = 8.6, max = 105, step=0.1, value=105)
lambda_min_w = ipywidgets.FloatText(description="λ_min", value=0.5)


show_choppers_button = ipywidgets.Button(description="Choppers", button_style="info")
show_sample_spectra_button = ipywidgets.Button(description="Spectra on sample", button_style="info")



cb_simulate = ipywidgets.Checkbox(value=False, description="Simulate pattern")


button = ipywidgets.Button(description="Generate peaks", button_style="success")

hkl_w = ipywidgets.Text(
    value='',
    placeholder='2 0 0',
    description='HKL:',
    disabled=False
)

button_hkl = ipywidgets.Button(description="Calc angles for HKL", button_style="success")

out = ipywidgets.Output()

# out = ipywidgets.interactive_output(
#     plot_hkl,
#     {
#         "gamma_min_max": gamma_min_max,
#         "nu_min_max": nu_min_max,
#         "lambda_min_max": lambda_min_max,
#         "h_min_max": h_min_max,
#         "k_min_max": k_min_max,
#         "l_min_max": l_min_max,
#     }
# )



In [ ]:
def get_magic_tof_model_from_widgets():
    psc_nu = psc_nu_dropdown.value
    psc_opening_angle = psc_opening_angle_floattext.value
    wavelength_band_min = lambda_min_w.value
    model = model_magic(
            psc_opening_angle=psc_opening_angle,
            psc_nu=psc_nu, 
            wavelength_band_min=wavelength_band_min,
            pulses=1,
            neutrons=10_000_000,
            )
    return model

def get_spectra_on_sample():
    model = get_magic_tof_model_from_widgets()
    res = model.run()
    data_flatten = res.detectors['Cave BM'].data.flatten(to='event2')
    data_reduced = data_flatten[~data_flatten.masks['blocked_by_others']]
    wavelength_min = data_reduced.coords["wavelength"].min().value
    wavelength_max = data_reduced.coords["wavelength"].max().value

    wavelength_point = int(numpy.round((wavelength_max-wavelength_min)/0.01, 0))+1
    bin_wavelength = scipp.linspace('wavelength', wavelength_min, wavelength_max, num=wavelength_point, unit='Å', endpoint=True)
    data_hist = data_reduced.hist(wavelength=bin_wavelength)    
    return data_hist, wavelength_min, wavelength_max


In [ ]:

# ---------------------------
# Callback
# ---------------------------

def show_choppers_clicked(_):
    with out:
        out.clear_output()
        model = get_magic_tof_model_from_widgets()
        for chopper_name, chopper in model.choppers.items():
            display(chopper.to_diskchopper())


def show_sample_spectra_clicked(_):
    with out:
        out.clear_output()
        data_hist, wavelength_min, wavelength_max= get_spectra_on_sample()
        fig = data_hist.plot()
        display(fig)


def compute_angles(_):
    with out:
        out.clear_output()
        cell_a = cell_a_w.value
        cell_b = cell_b_w.value
        cell_c = cell_c_w.value

        cell_alpha = numpy.deg2rad(cell_alpha_w.value)
        cell_beta  = numpy.deg2rad(cell_beta_w.value)
        cell_gamma = numpy.deg2rad(cell_gamma_w.value)

        ea_alpha = numpy.deg2rad(ea_alpha_w.value)
        ea_beta  = numpy.deg2rad(ea_beta_w.value)
        ea_gamma = numpy.deg2rad(ea_gamma_w.value)

        sample_omega = numpy.deg2rad(omega_w.value)
        sample_chi   = numpy.deg2rad(chi_w.value)
        sample_phi   = numpy.deg2rad(phi_w.value)

        # --- compute matrices ---
        m_b = calc_b_matrix(cell_a, cell_b, cell_c,
                            cell_alpha, cell_beta, cell_gamma)

        m_u = calc_orientation_matrix(ea_alpha, ea_beta, ea_gamma)
        m_ub = m_u @ m_b

        m_r = calc_sample_rotation(sample_omega, sample_chi, sample_phi)
        s_hkl = hkl_w.value
        try:
            (h,k,l) = (float(hh) for hh in s_hkl.strip().split()[:3])
        except:
            s_hkl = hkl_w.placeholder
            (h,k,l) = (float(hh) for hh in s_hkl.strip().split()[:3])

        gamma, nu, wavelength = calc_gamma_nu_wavelength_for_hkl(h,k,l, m_ub, m_r)
        print(f'# Peak ({h:.1f} {k:.1f} {l:.1f})') 
        print(f"\nUB matrix: \n{m_ub[0,0]:9.5f}{m_ub[0,1]:9.5f}{m_ub[0,1]:9.5f}\n{m_ub[1,0]:9.5f}{m_ub[1,1]:9.5f}{m_ub[1,2]:9.5f}\n{m_ub[2,0]:9.5f}{m_ub[2,1]:9.5f}{m_ub[2,2]:9.5f}")
        print(f"\nRotation matrix: \n{m_r[0,0]:9.5f}{m_r[0,1]:9.5f}{m_r[0,1]:9.5f}\n{m_r[1,0]:9.5f}{m_r[1,1]:9.5f}{m_r[1,2]:9.5f}\n{m_r[2,0]:9.5f}{m_r[2,1]:9.5f}{m_r[2,2]:9.5f}")
        print(f"\nGamma is {gamma.squeeze():.2f} deg.\nNu is {nu.squeeze():.2f} deg.\nWavelength is {wavelength.squeeze():.5f} Ang.")


def on_file_selected(_):
    file_path = fc.selected

    with out:
        out.clear_output()
        if file_path is None or not os.path.isfile(file_path):
            # --- read inputs ---
            pass
        else:
            rcif_obj = cryspy.file_to_globaln(file_path)          
            crystal = [hh for hh in rcif_obj.items if isinstance(hh, cryspy.Crystal)][0]
            ucp = crystal.get_dictionary()['unit_cell_parameters']
            cell_a_w.value = ucp[0]
            cell_b_w.value = ucp[1]
            cell_c_w.value = ucp[2]
            cell_alpha_w.value = numpy.rad2deg(ucp[3])
            cell_beta_w.value = numpy.rad2deg(ucp[4])
            cell_gamma_w.value = numpy.rad2deg(ucp[5])

    return


def compute(_):
    with out:
        global data
        out.clear_output()

        file_path = fc.selected
        flag_crystal = False
        if file_path is None or not os.path.isfile(file_path):
            # --- read inputs ---
            pass
        else:
            try:
                rcif_obj = cryspy.file_to_globaln(file_path)            
                crystal = [hh for hh in rcif_obj.items if isinstance(hh, cryspy.Crystal)][0]
                flag_crystal = True
            except:
                flag_crystal = False

        cell_a = cell_a_w.value
        cell_b = cell_b_w.value
        cell_c = cell_c_w.value
        cell_alpha = numpy.deg2rad(cell_alpha_w.value)
        cell_beta = numpy.deg2rad(cell_beta_w.value)
        cell_gamma = numpy.deg2rad(cell_gamma_w.value)



        ea_alpha = numpy.deg2rad(ea_alpha_w.value)
        ea_beta  = numpy.deg2rad(ea_beta_w.value)
        ea_gamma = numpy.deg2rad(ea_gamma_w.value)

        sample_omega = numpy.deg2rad(omega_w.value)
        sample_chi   = numpy.deg2rad(chi_w.value)
        sample_phi   = numpy.deg2rad(phi_w.value)

        da_spectra_hist, lambda_min, lambda_max = get_spectra_on_sample()

        # --- compute limits ---
        hmax = int(numpy.round(2 * cell_a / lambda_min,0))
        kmax = int(numpy.round(2 * cell_b / lambda_min,0))
        lmax = int(numpy.round(2 * cell_c / lambda_min,0))

        # --- compute matrices ---
        m_b = calc_b_matrix(cell_a, cell_b, cell_c,
                            cell_alpha, cell_beta, cell_gamma)

        m_u = calc_orientation_matrix(ea_alpha, ea_beta, ea_gamma)
        m_ub = m_u @ m_b

        m_r = calc_sample_rotation(sample_omega, sample_chi, sample_phi)

        # --- generate peak data ---
        data = generate_peak_data(
            UB=m_ub,
            R=m_r,
            hmax=hmax, kmax=kmax, lmax=lmax,
            lambda_min=lambda_min,
            lambda_max=lambda_max
        )

        mask = numpy.logical_and(data[:,3] >= gamma_min_w.value, data[:,3] <= gamma_max_w.value)
        data = data[mask]
        mask = numpy.logical_and(data[:,4] >= nu_min_w.value, data[:,4] <= nu_max_w.value)
        data = data[mask]
        
        cos_tth = numpy.cos(numpy.radians(data[:,3]))*numpy.cos(numpy.radians(data[:,4]))
        if flag_crystal:
            fn = crystal.calc_f_nucl(numpy.stack([data[:,0], data[:,1], data[:,2]], axis=0))
            fn_sq = numpy.square(numpy.abs(fn))
            np_spectra = numpy.interp(data[:,5], scipp.midpoints(da_spectra_hist.coords['wavelength']).values, da_spectra_hist.data.values, left=0., right=0.)

            th = 0.5*numpy.acos(cos_tth)
            d_hkl = data[:,5]/(2*numpy.sin(th))
            lf = numpy.pow(d_hkl,4)/numpy.sin(th)
            iint = lf*fn_sq * np_spectra
            data = numpy.column_stack((data, iint))

        size = 4
        if data.shape[1]>=7:
            mask = data[:,6] >= 0.00001*data[:,6].max()
            data = data[mask]
            size = 5. * size * data[:,6]/data[:,6].max()

        # --- output ---
        if data.shape[1]>=7:
            if cb_simulate.value:
                gamma = numpy.linspace(gamma_min_w.value, gamma_max_w.value, num=int((gamma_max_w.value-gamma_min_w.value)/0.2),endpoint=True)
                nu= numpy.linspace(nu_min_w.value, nu_max_w.value, num=int((nu_max_w.value-nu_min_w.value)/0.2),endpoint=True)
                gamma_grid, nu_grid = np.meshgrid(gamma, nu)
                pattern = simulate_detector(data[:,6], data[:,3], data[:,4], gamma_grid,nu_grid, sigma=0.25)
                fig = plot_pattern(pattern, gamma_grid,nu_grid)
                size = 0.2
                color = 'white'
                color_text = 'white'
            else:
                fig = None
                color = hkl_to_rgb(data[:,0], data[:,1], data[:,2])
                color_text = 'black'

            labels = [f"({h:.0f} {k:.0f} {l:.0f})" for h,k,l in zip(data[:,0], data[:,1], data[:,2])]
            fig = plot_scatters_with_labels(data[:,3], data[:,4], labels, color=color, size=size, n_visible=20, fig=fig, color_text=color_text)
            df = pandas.DataFrame(data, columns=['H', 'K', 'L', 'gamma', 'nu', 'wavelength', 'iint'])

        else:
            labels = [f"({h:.0f} {k:.0f} {l:.0f})" for h,k,l in zip(data[:,0], data[:,1], data[:,2])]
            np_rgb = hkl_to_rgb(data[:,0], data[:,1], data[:,2])
            fig = plot_scatters_with_labels(data[:,3], data[:,4], labels, color=np_rgb, size=size, n_visible=20)
            df = pandas.DataFrame(data, columns=['H', 'K', 'L', 'gamma', 'nu', 'wavelength'])
        fig.axes[0].set_xlim((gamma_min_w.value, gamma_max_w.value))
        fig.axes[0].set_ylim((nu_min_w.value, nu_max_w.value))

        display(fig.canvas)
        display(df)

 


In [ ]:
fc.register_callback(on_file_selected)

show_choppers_button.on_click(show_choppers_clicked)
show_sample_spectra_button.on_click(show_sample_spectra_clicked)
button.on_click(compute)
button_hkl.on_click(compute_angles)

In [ ]:
ui = ipywidgets.VBox([
    fc, 
    ipywidgets.HTML("<h3>Diffraction Peaks on detector</h3>"),
    ipywidgets.HTML("<h4>Unit cell parameters</h4>"),
    ipywidgets.HBox([cell_a_w, cell_b_w, cell_c_w]),
    ipywidgets.HBox([cell_alpha_w, cell_beta_w, cell_gamma_w]),
    ipywidgets.HTML("<h4>Sample Rotation</h4>"),
    ipywidgets.HBox([omega_w, chi_w, phi_w]),
    ipywidgets.HTML("<h4>Chopper settings</h4>"),
    ipywidgets.HBox([psc_nu_dropdown, psc_opening_angle_floattext, lambda_min_w]),
    ipywidgets.HBox([show_choppers_button, show_sample_spectra_button,]),
    ipywidgets.HTML("<h4>Detector Range</h4>"),
    ipywidgets.HBox([gamma_min_w, gamma_max_w,]),
    ipywidgets.HBox([nu_min_w, nu_max_w,]),
    ipywidgets.HTML("<h4>Sample Orientation (Euleur angles)</h4>"),
    ipywidgets.HBox([ea_alpha_w, ea_beta_w, ea_gamma_w]),
    ipywidgets.HTML("<h4> </h4>"),
    ipywidgets.HBox([button, cb_simulate]),
    ipywidgets.HBox([button_hkl, hkl_w]),
    out,
])

display(ui)